In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4
from dask import array as da
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import BoundaryNorm

import geopandas as gp

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.util import add_cyclic_point

import glob
from tqdm import tqdm

import re

In [2]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="60GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=60GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="00:13:00",  # Amount of wall time
    interface="ext",  # Interface to use
)
# cluster.scale(96)
# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=4) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

# show the client that you have been assigned, you can click on the link and it will show you 
# a dashboard with all the tasks that have to be performed to do your calculation
client

/glade/work/jtcohen/envs/lib/python3.10/site-packages/distributed/node.py:179: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39467 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/39467/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/39467/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.179:41003,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/39467/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [4]:
rad_val = 'andom'
nens = 20

MODE_output = f'/glade/derecho/scratch/jtcohen/MODE_files/output_r{rad_val}'
MODE_save = '/glade/work/jtcohen/MODE_files_final/saved_output'

variables = [
    # 'fcst_obj_raw',
    # 'fcst_obj_id',
    'fcst_clus_id',
    # 'obs_obj_raw',
    # 'obs_obj_id',
    'obs_clus_id'
]

def preprocess(ds):
    return ds[variables]

In [5]:
%%time
firstyear = 2004 # 1989
lastyear = 2018 # 2003
inits = pd.date_range(f'{firstyear}-02-01', f'{lastyear}-11-01', freq='QS-FEB')
ms = np.arange(nens) + 1
leads = np.arange(24)
chunks = {'lead': 24, 'member': 20, 'init': 1}
fnames = [[sorted(glob.glob(f'{MODE_output}/i{init.strftime("%Y%m%d")}/mode_m{m:02d}E*.nc')) for m in ms] for init in inits]
# np.shape(fnames)
ds = xr.open_mfdataset(fnames, combine='nested', concat_dim=['init', 'member', 'lead'], compat='override', coords='minimal', preprocess=preprocess, parallel=True, decode_cf=False).chunk(chunks)
ds['lead'] = leads
ds['member'] = ms
ds['init'] = inits

CPU times: user 3min 59s, sys: 4 s, total: 4min 3s
Wall time: 5min 17s


In [ ]:
%%time
ds.load().to_netcdf(f'{MODE_save}/object_footprints_r{rad_val}_{firstyear}-{lastyear}.nc')